### Exploratory Data Analysis

## Scope and Data Context

This notebook performs exploratory data analysis (EDA) on a cleaned synthetic healthcare dataset
generated by `01_data_prep.ipynb`.

Each row represents a single patient visit, capturing visit-level cost, services utilized,
and temporal information. All data preparation and validation steps were completed upstream;
no additional cleaning is performed in this notebook.

The goal of this EDA is to characterize distributional properties, identify key relationships
between variables, and surface patterns that may inform downstream statistical analysis
or modeling decisions.

**Objectives:**
- Characterize distributional properties of key measures
- Identify relationships between variables
- Surface patterns that may inform downstream modeling decisions

Unless otherwise noted, all analyses are descriptive and non-inferential.




## 1) Variable Classification

This section classifies variables into analytically meaningful types: quantitative measures 
(continuous/discrete), categorical attributes (nominal/ordinal), and datetime fields.

These classifications inform downstream decisions on appropriate summary statistics, 
visualizations, and statistical tests.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
import os
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

# Paths
DATA_PATH = '../data/processed/integrated_output.csv'
RESULTS_DIR = '../results/figures'

os.makedirs(RESULTS_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")

In [ ]:
print(df.info())
print(df.head(5))

To better reflect visit-level economic impact, a revenue metric is constructed
as the product of unit amount and service quantity.

In [ ]:
# Add Revenue column
df['Revenue'] = df['Amount'] * df['Quantity']

# Drop index column if it exists
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

df.head(5)

All variables are classified according to their analytical roles. This
classification reflects how each variable is intended to be interpreted and analyzed,
rather than its raw storage type.

**The resulting variable roles are summarized in the table below.**

| Variable | Measure/Category | Continuous/Discrete | Nominal/Ordinal |
| :--- | :--- | :--- | :--- |
| Visit_ID | Category | N/A | Nominal |
| Patient_ID | Category | N/A | Nominal |
| Clinic_ID | Category | N/A | Nominal |
| Visit_Date | Category | N/A | Ordinal |
| Procedure_Code | Category | N/A | Nominal |
| Amount | Measure | Continuous | N/A |
| Quantity | Measure | Discrete | N/A |
| is_refund | Category | N/A | Nominal |
| is_outlier | Category | N/A | Nominal |
| email | Category | N/A | Nominal |
| birth_date | Category | N/A | Ordinal |
| region_code | Category | N/A | Nominal |
| Procedure_Name | Category | N/A | Nominal |
| Specialty | Category | N/A | Nominal |
| Standard_Cost | Measure | Continuous | N/A |
| is_unrealistic_cost | Category | N/A | Nominal |
| Region_Code | Category | N/A | Nominal |
| Region_Name | Category | N/A | Nominal |
| Revenue | Measure | Continuous | N/A |

In [ ]:
# Check current data types
print('Current Data Types:')
print(df.dtypes)

The categorical & datetime data types are modified in the DataFrame:


In [ ]:
# Make a clean copy of the current df
df_clean = df.copy()

# Categorical variables
categorical_vars = [
    'Visit_ID', 'Patient_ID', 'Clinic_ID', 'Procedure_Code',
    'is_refund', 'is_outlier', 'email', 'region_code',
    'Procedure_Name', 'Specialty', 'is_unrealistic_cost',
    'Region_Code', 'Region_Name'
]

for var in categorical_vars:
  if var in df_clean.columns:
    df_clean[var] = df_clean[var].astype('category')

# Treat as datetime to preserve ordinal nature while enabling date operations
ordinal_date_vars = ['visit_Date', 'birth_date']
for var in ordinal_date_vars:
    if var in df_clean.columns:
        df_clean[var] = pd.to_datetime(df_clean[var])

In [ ]:
print('Updated Data Types:')
print(df_clean.dtypes)

## 2) Univariate Analysis


This section summarizes the distribution of measure
variables using descriptive statistics. The table below provides measures of
central tendency, dispersion, and distribution shape.


In [ ]:
# Visual style configuration
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'primary': '#4C72B0',
    'mean': '#C44E52',
    'median': '#55A868',
    'box': '#8DA0CB'
}

measure_vars = ['Amount', 'Quantity', 'Standard_Cost', 'Revenue']

In [ ]:
# =============================================================================
# Descriptive Statistics Table
# =============================================================================
def create_descriptive_stats(df, measure_vars):
    """
    Generate comprehensive descriptive statistics for measure variables."""
    detailed_stats = []

    for col in measure_vars:
        data = df[col].dropna()

        detailed_stats.append({
            'Variable': col,
            'Count': len(data),
            'Mean': f"{data.mean():.2f}",
            'Median': f"{data.median():.2f}",
            'Min': f"{data.min():.2f}",
            'Max': f"{data.max():.2f}",
            'Std Dev': f"{data.std():.2f}",
            'Variance': f"{data.var():.2f}",
            'CV %': f"{(data.std() / data.mean()) * 100:.2f}%" if data.mean() != 0 else 'N/A',
            'Q1': f"{data.quantile(0.25):.2f}",
            'Q3': f"{data.quantile(0.75):.2f}",
            'IQR': f"{data.quantile(0.75) - data.quantile(0.25):.2f}",
            'Skewness': f"{stats.skew(data):.2f}",
            'Kurtosis': f"{stats.kurtosis(data):.2f}",
            'Missing': df[col].isnull().sum()
        })

    return pd.DataFrame(detailed_stats)

descriptive_table = create_descriptive_stats(df_clean, measure_vars)
descriptive_table.to_csv(f'{RESULTS_DIR}/descriptive_stats.csv', index=False)
display(descriptive_table)

In [ ]:
# =============================================================================
# Distribution Plots - Histogram with Boxplot
# =============================================================================

"""
Marginal distributions of key quantitative variables using histograms
and boxplots. The goal is to assess distribution shape, skewness, and
the presence of extreme values.
"""

for column in measure_vars:
    data = df_clean[column].dropna()

    # Statistics
    skewness = stats.skew(data)
    kurtosis = stats.kurtosis(data)
    mean_val = data.mean()
    median_val = data.median()

    # Create figure: histogram + boxplot
    fig, (ax_hist, ax_box) = plt.subplots(
        1, 2, figsize=(12, 5),
        gridspec_kw={'width_ratios': [3, 1]}
    )

    # Histogram with KDE
    sns.histplot(data=data, kde=True, color=COLORS['primary'],
                 alpha=0.7, ax=ax_hist, edgecolor='white', linewidth=0.5)

    # Mean and median lines
    ax_hist.axvline(mean_val, color=COLORS['mean'], linestyle='--', lw=2)
    ax_hist.axvline(median_val, color=COLORS['median'], linestyle=':', lw=2)

    # Labels
    ax_hist.set_title(f'Distribution of {column}', fontsize=14, fontweight='bold')
    ax_hist.set_xlabel(column, fontsize=11)
    ax_hist.set_ylabel('Frequency', fontsize=11)

    # Legend (bottom left - away from stats box)
    legend_elements = [
        Line2D([0], [0], color=COLORS['mean'], linestyle='--', lw=2,
               label=f'Mean: {mean_val:,.2f}'),
        Line2D([0], [0], color=COLORS['median'], linestyle=':', lw=2,
               label=f'Median: {median_val:,.2f}')
    ]
    ax_hist.legend(handles=legend_elements, loc='upper left', fontsize=9)

    # Stats box (upper right - no overlap)
    stats_text = f'Skewness: {skewness:.2f}\nKurtosis: {kurtosis:.2f}'
    ax_hist.text(0.98, 0.95, stats_text, transform=ax_hist.transAxes,
                 ha='right', va='top', fontsize=10,
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                          edgecolor='gray', alpha=0.9))

    # Boxplot
    sns.boxplot(y=data, color=COLORS['box'], ax=ax_box, width=0.5)
    ax_box.set_title('Boxplot', fontsize=12, fontweight='bold')
    ax_box.set_ylabel('')

    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/dist_{column.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# Log-Transformed Distributions (Skewed Variables Only)
# =============================================================================
"""
For highly skewed variables, log transformation can reveal underlying
patterns obscured by extreme values. This is particularly relevant for
Amount and Revenue which show strong positive skew.
"""

# Identify skewed variables (|skewness| > 1)
skewed_vars = []
for col in measure_vars:
    data = df_clean[col].dropna()
    if abs(stats.skew(data)) > 1:
        skewed_vars.append(col)

print(f"Varaibles with |skewness| > 1: {skewed_vars}\n")

for column in skewed_vars:
    data = df_clean[column].dropna()

    # Handle negative values for log transformation
    min_val = data.min()
    if min_val <= 0:
        shift = abs(min_val) + 1
        log_data = np.log(data + shift)
        transform_note = f'log(x + {shift})' # Define transform_note here
    else:
        log_data = np.log1p(data)
        transform_note = 'log(x + 1)'

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Original distribution
    sns.histplot(data=data, kde=True, color=COLORS['primary'],
                 alpha=0.7, ax=ax1, edgecolor='white')
    ax1.axvline(data.mean(), color=COLORS['mean'], linestyle='--', lw=2,
                label=f'Mean: {data.mean():,.2f}')
    ax1.axvline(data.median(), color=COLORS['median'], linestyle=':', lw=2,
                label=f'Median: {data.median():,.2f}')
    ax1.set_title(f'{column} - Original Scale', fontsize=13, fontweight='bold')
    ax1.set_xlabel(column, fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.legend(loc='upper right', fontsize=9)

    skew_orig = stats.skew(data)
    kurt_orig = stats.kurtosis(data)
    ax1.text(0.02, 0.95, f'Skew: {skew_orig:.2f}\nKurt: {kurt_orig:.2f}',
             transform=ax1.transAxes, ha='left', va='top', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

    # Log-transformed distribution
    sns.histplot(data=log_data, kde=True, color='#66C2A5',
                 alpha=0.7, ax=ax2, edgecolor='white')
    ax2.axvline(log_data.mean(), color=COLORS['mean'], linestyle='--', lw=2,
                label=f'Mean: {log_data.mean():.2f}')
    ax2.axvline(log_data.median(), color=COLORS['median'], linestyle=':', lw=2,
                label=f'Median: {log_data.median():.2f}')
    ax2.set_title(f'{column} - Log Transformed ({transform_note})',
                  fontsize=13, fontweight='bold')
    ax2.set_xlabel(f'log({column})', fontsize=11)
    ax2.set_ylabel('Frequency', fontsize=11)
    ax2.legend(loc='upper right', fontsize=9)

    skew_log = stats.skew(log_data)
    kurt_log = stats.kurtosis(log_data)
    ax2.text(0.02, 0.95, f'Skew: {skew_log:.2f}\nKurt: {kurt_log:.2f}',
             transform=ax2.transAxes, ha='left', va='top', fontsize=10,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

    plt.suptitle(f'Effect of Log Transformation on {column}',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/dist_{column.lower()}_log.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# Combined Boxplot Grid
# =============================================================================

"""
Side-by-side boxplot comparison to identify which variables have the
most extreme outliers and compare relative spread across measures.
"""

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel() # Flatten the 2x2 array of axes to easily iterate

for i, column in enumerate(measure_vars):
    data = df_clean[column].dropna()

    # Calculate outlier count (IQR method)
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    outliers = data[(data < Q1 - 1.5 * IQR) | (data > Q3 + 1.5 * IQR)]

    sns.boxplot(data=df_clean, y=column, color=COLORS['box'], ax=axes[i],
                width=0.5, flierprops={'marker': 'o', 'markersize': 4})

    axes[i].set_title(f'Boxplot of {column}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel(column, fontsize=11)

    # Outlier count annotation
    axes[i].text(0.95, 0.95, f'Outliers: {len(outliers)}',
                 transform=axes[i].transAxes, ha='right', va='top',
                 fontsize=10, bbox=dict(boxstyle='round,pad=0.3',
                                        facecolor='lightyellow', alpha=0.9))

plt.suptitle('Boxplot Comparison of Measure Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/boxplot_grid.png', dpi=150, bbox_inches='tight')
plt.show()

### Distribution Shape of Each Variable:
**Amount:** Highly right-skewed (skewness ~5.6) with heavy tails (kurtosis ~47).
The mean (159) exceeds the median considerably, pulled by high-value outliers.
This pattern is typical for transaction data where most values cluster low
with occasional large amounts. Negative values represent refunds.

**Quantity:** Moderately right-skewed. Most procedures involve single units,
with occasional bulk quantities creating the right tail. The discrete nature
of this variable (whole numbers) is expected for procedure counts.

**Standard_Cost:** Approximately symmetric (skewness ~0.15) with lighter tails
than normal (negative kurtosis). This reflects standardized pricing tiers
across procedure types—costs are predetermined rather than variable.

**Revenue:** Highly right-skewed, similar to Amount. As the product of
Amount × Quantity, it inherits and amplifies the skewness from both inputs.
Negative values from refunds create a minor left tail.

### Outlier Assessment:

Revenue shows the most extreme outliers, followed by Amount. These are
driven by the multiplicative relationship (high amount × high quantity)
and should be investigated for data quality vs. legitimate high-value cases.

---

**Charts for categories**


Variables `email`, `Patient_ID`, and `Visit_ID` were excluded from bar chart visualization as they are unique identifiers with high cardinality, making frequency distributions uninformative.

In [ ]:
# =============================================================================
# Distribution Plots - Categorical Variables
# =============================================================================

# Exclude variables with too many unique values for bar charts
categorical_vars_to_plot = [col for col in categorical_vars
                            if col not in ['email', 'Patient_ID', 'Visit_ID']]

for col in categorical_vars_to_plot:
    n_unique = df_clean[col].nunique()

    # Horizontal bars for many categories, vertical for few
    if n_unique > 8:
        fig, ax = plt.subplots(figsize=(10, max(5, n_unique * 0.4)))
        order = df_clean[col].value_counts().index
        sns.countplot(data=df_clean, y=col, order=order,
                      color=COLORS['primary'], ax=ax)
        ax.set_xlabel('Count', fontsize=11)
        ax.set_ylabel(col, fontsize=11)
        ax.bar_label(ax.containers[0], fmt='%d', padding=3)
    else:
        fig, ax = plt.subplots(figsize=(8, 5))
        order = df_clean[col].value_counts().index
        sns.countplot(data=df_clean, x=col, order=order,
                      color=COLORS['primary'], ax=ax)
        ax.set_xlabel(col, fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        plt.xticks(rotation=45, ha='right')
        ax.bar_label(ax.containers[0], fmt='%d', padding=3)

    ax.set_title(f'Distribution of {col}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/cat_{col.lower()}.png',dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Listing no. of missing values
missing_values = df_clean.isnull().sum()
missing_values_table = pd.DataFrame(missing_values, columns=['Missing Values'])
print(missing_values_table)

### Missing Value Notes

- **Region-related columns** (`region_code`, `Region_Code`, `Region_Name`): 77 missing values each.
  These are linked - `Region_Name` was joined from `region_code`, so missingness propagates.
  Represents ~25.67% of records with unknown patient region.

- **Patient demographics** (`name`, `email`, `birth_date`): 8-15 missing values.
  Minor missingness (<1%), likely incomplete registration records.

- **Clinic_ID**: 62 missing. May indicate visits not yet assigned to a clinic or data entry gaps.

- **Standard_Cost**: 10 missing. Procedures without established pricing - worth investigating
  which `Procedure_Code` values these correspond to.

No imputation performed for this EDA phase; missing values excluded from relevant analyses.

## 3)  Bivariate Analysis


3.1)  Create and display below a correlation matrix of the measures in your dataset.
Comment below on the correlation matrix. What correlations exist? Do you see anything unexpected?

In [ ]:
# Create a measure variable dataframe
measure_df = df_clean[measure_vars]

# Calculate the correlation matrix
correlation_matrix = measure_df.corr()

print(correlation_matrix)

In [ ]:
# heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix - Measure Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

**Correlation Matrix Interpretation:**

| Pair | r | Assessment |
|------|---|------------|
| Amount ↔ Revenue | 0.86 | Strong positive — expected, as Revenue = Amount × Quantity |
| Quantity ↔ Revenue | 0.25 | Weak positive — Quantity contributes to Revenue but less than Amount |
| Quantity ↔ Amount | -0.02 | Near zero — no meaningful linear relationship |
| Standard_Cost ↔ Amount | 0.04 | **Unexpectedly weak** — suggests actual charges deviate significantly from standard pricing (discounts, adjustments, negotiated rates) |
| Standard_Cost ↔ Revenue | 0.02 | Near zero — Standard_Cost is a poor predictor of actual revenue |

**Key insight:** The disconnect between Standard_Cost and actual Amount/Revenue suggests
this synthetic data includes realistic pricing variability (insurance adjustments, discounts)
rather than a simple cost = charge relationship.

### Revenue Distribution by Categorical Variables

To explore relationships between categorical attributes and the target measure,
side-by-side boxplots compare Revenue distributions across category levels.
This reveals which categories are associated with higher/lower revenue and
identifies potential grouping effects for downstream modeling.

In [ ]:
# =============================================================================
# Revenue Distribution by Categorical Variables
# =============================================================================

categorical_vars_to_plot = [col for col in categorical_vars
                            if col not in ['email', 'Patient_ID', 'Visit_ID']]

for col in categorical_vars_to_plot:
    n_unique = df_clean[col].nunique()

    # Horizontal layout for many categories, vertical for few
    if n_unique > 6:
        fig, ax = plt.subplots(figsize=(10, max(5, n_unique * 0.5)))
        order = (df_clean.groupby(col, observed=True)['Revenue']
                 .median().sort_values(ascending=False).index)
        sns.boxplot(data=df_clean, y=col, x='Revenue', order=order,
                    color=COLORS['box'], ax=ax)
        ax.set_ylabel(col, fontsize=11)
        ax.set_xlabel('Revenue', fontsize=11)
    else:
        fig, ax = plt.subplots(figsize=(10, 6))
        order = (df_clean.groupby(col, observed=True)['Revenue']
                 .median().sort_values(ascending=False).index)
        sns.boxplot(data=df_clean, x=col, y='Revenue', order=order,
                    color=COLORS['box'], ax=ax)
        ax.set_xlabel(col, fontsize=11)
        ax.set_ylabel('Revenue', fontsize=11)
        plt.xticks(rotation=45, ha='right')

    ax.set_title(f'Revenue by {col}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/bivar_{col.lower()}_revenue.png', dpi=150, bbox_inches='tight')
    plt.show()

**Comments on the Boxplots (Categorical Variables vs. Revenue):**

*   **is_refund:** As expected, there is a clear distinction in Revenue based on the `is_refund` variable. The 'True' category (refunds) shows primarily negative Revenue values, while the 'False' category (non-refunds) shows positive Revenue values. This confirms the direct relationship between this categorical variable and the sign of the Revenue.
*   **is_outlier:** The boxplot for `is_outlier` is interesting. The "False" category shows a typical distribution of Revenue, the "True" category (outliers) displays a much wider spread and likely includes both very high and potentially very low (or negative) Revenue values that were flagged as outliers during the initial data cleaning.
*   **is_unrealistic_cost:** Similar to `is_outlier`, the `is_unrealistic_cost` variable also highlights differences in Revenue. The "True" category, representing cases with unrealistic standard costs, appears to be associated with a wider range of Revenue values, potentially including some extreme values that deviate from the norm.
*   **Clinic_ID:** The boxplots for `Clinic_ID` reveal variations in the distribution of Revenue across different clinics. Some clinics might have higher median Revenue or a wider spread of Revenue values compared to others. This could indicate differences in pricing, types of procedures performed, or patient volume across clinics.
*   **Procedure_Code / Procedure_Name:** These boxplots show how Revenue varies depending on the specific procedure performed. Different procedures have different costs and quantities, which directly impact the Revenue generated. You can observe which procedures tend to generate higher or lower Revenue and which ones have a wider variability in Revenue.
*   **region_code / Region_Name:** The boxplots for region indicate potential geographical variations in Revenue. Some regions might have higher or lower typical Revenue per visit, which could be influenced by factors like local pricing, economic conditions, or the types of services in demand in that region.
*   **Specialty:** The boxplots for `Specialty` highlight differences in Revenue based on the medical specialty. Some specialties might inherently involve more expensive procedures or a higher volume of services, leading to higher Revenue distributions compared to others.


### Pairwise Relationships Among Measures

Scatterplot matrices visualize bivariate relationships between all measure
variables, highlighting linear associations, non-linear patterns, and potential
outlier influence. This complements the correlation matrix by revealing
distributional structure that correlation coefficients alone cannot capture.

In [ ]:
from ast import FunctionType
# Scatterplots of each measure vs Revenue
fig, axes = plt.subplots(1, 3, figsize=(15,5))
other_measures = ['Amount', 'Quantity', 'Standard_Cost']

for i, col in enumerate(other_measures):
    sns.scatterplot(data=df_clean, x=col, y='Revenue',
                    alpha=0.6, color=COLORS['primary'], ax=axes[i])
    axes[i].set_title(f'{col} vs Revenue', fontsize=13, fontweight='bold')
    axes[i].set_xlabel(col, fontsize=11)
    axes[i].set_ylabel('Revenue', fontsize=11)

    # Add correlation annotation
    r = df_clean[[col, 'Revenue']].corr().iloc[0, 1]
    axes[i].text(0.05, 0.95, f'r = {r:.2f}', transform=axes[i].transAxes,
                 ha='left', va='top', fontsize=10,
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

plt.suptitle('Measure Variables vs Revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/scatter_vs_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

### Multicollinearity Assessment

Variance Inflation Factor (VIF) quantifies how much the variance of a regression
coefficient is inflated due to collinearity with other predictors. VIF > 5
indicates moderate concern; VIF > 10 suggests severe multicollinearity requiring
variable exclusion or dimensionality reduction.

In [ ]:
measure_vars = ['Amount', 'Quantity', 'Standard_Cost', 'Revenue']

# Create a temporary DataFrame w/ measure variables and drop rows with missing values
measure_df_cleaned = df_clean[measure_vars].dropna()

# Calculate VIF for each variable
vif_data = pd.DataFrame()
vif_data["variable"] = measure_vars
vif_data["VIF"] = [vif(measure_df_cleaned.values, i)
  for i in range(measure_df_cleaned.shape[1])]

print(vif_data)